# What is Grid Search CV?

Grid Search CV (Cross-Validation) is a brute force method to find the best hyperparameters for a model.

- You define a grid (set) of possible values for hyperparameters.

- The algorithm trains and evaluates the model for every possible combination.

- Uses cross-validation to measure performance reliably.

- Finally picks the combination that performs best.

- **Simple but can be very slow if you have many parameters.**



## The Math Behind It

Suppose we have:

- Model $f(x;\theta)$ with parameters $\theta$.

- Hyperparameters $h = (h_1, h_2, ..., h_k)$.

- Grid $G = \{h_{11}, h_{12}, ..., h_{1m}\} \times \{h_{21}, ..., h_{2n}\}$.



For each $h \in G$:

- Train the model with hyperparameters $h$.

- Compute cross-validation score:

  $$CV(h) = \frac{1}{K} \sum_{i=1}^K Accuracy_i(h)$$

  (where $K$ = number of folds)

- Best hyperparameters:

  $$h^* = \arg\max_{h \in G} CV(h)$$



## Example

Suppose we’re tuning an SVM:

- Hyperparameters:

  - Kernel = {linear, rbf}

  - C = {0.1, 1, 10}

  - Gamma = {0.01, 0.1}

- Grid = all possible combos = $2 \times 3 \times 2 = 12$



Grid Search CV will:

- Train SVM for each of the 12 settings.

- Use K-fold CV (say K=5) → 5 evaluations per setting.

- Pick best performing combination.







## Visual (Conceptual)

Imagine a 2D grid for two hyperparameters (C vs gamma).
- Each point = one model training + CV.
- We try all points, and pick the “brightest” (best score).



## Comparison with Bayesian Optimization

- **Grid Search CV:** Tries all combos blindly → exhaustive but slow.

- **Bayesian Optimization:** Builds a probability model, explores smartly.



In [47]:
import pandas as pd

In [30]:
df=pd.read_csv("balanced_fraud_detection_data.csv")

In [36]:
from sklearn.model_selection import train_test_split

X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [46]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier 
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [ ]:
models = {
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7,
            'weights': ['uniform', 'distance'],
            'algorithm': ['auto', 'ball_tree'],
            'leaf_size': [30, 50]
        }
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5],
            'criterion': ['gini', 'entropy']
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(solver='liblinear', random_state=42),
        'params': {
            'C': [0.1, 1, 10],
            'penalty': ['l1', 'l2'],
            'fit_intercept': [True, False],
            'max_iter': [100, 200]
        }
    }
}

results = {}
for name, mp in models.items():
    print(f"\nRunning GridSearchCV for {name}...")
    grid = GridSearchCV(mp['model'], mp['params'], cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    results[name] = {
        'best_score': grid.best_score_,
        'best_params': grid.best_params_,
        'test_score': grid.score(X_test, y_test)
    }
    print(f"Best CV Score: {grid.best_score_:.4f}")
    print(f"Best Params: {grid.best_params_}")
    print(f"Test Score: {grid.score(X_test, y_test):.4f}")

print("\nSummary of Results:")
for name, res in results.items():
    print(f"{name}: Best CV Score={res['best_score']:.4f}, Test Score={res['test_score']:.4f}, Best Params={res['best_params']}")


Running GridSearchCV for KNN...
Best CV Score: 0.9280
Best Params: {'algorithm': 'auto', 'leaf_size': 30, 'n_neighbors': 7, 'weights': 'distance'}
Test Score: 0.9210

Running GridSearchCV for RandomForest...
Best CV Score: 0.9617
Best Params: {'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Test Score: 0.9560

Running GridSearchCV for LogisticRegression...
Best CV Score: 0.8839
Best Params: {'C': 0.1, 'fit_intercept': True, 'max_iter': 100, 'penalty': 'l2'}
Test Score: 0.8740

Summary of Results:
KNN: Best CV Score=0.9280, Test Score=0.9210, Best Params={'algorithm': 'auto', 'leaf_size': 30, 'n_neighbors': 7, 'weights': 'distance'}
RandomForest: Best CV Score=0.9617, Test Score=0.9560, Best Params={'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
LogisticRegression: Best CV Score=0.8839, Test Score=0.8740, Best Params={'C': 0.1, 'fit_intercept': True, 'max_iter': 100, 'penalty': 'l2'}


In [ ]:
# it took 1m 40 sec for grid search,it takes a lot longer for bayesian optimization and random search

In [41]:
rf = RandomForestClassifier(**results['RandomForest']['best_params'], random_state=42)
rf.fit(X_train, y_train)

RandomForestClassifier(criterion='entropy', random_state=42)

In [42]:
y_pred = rf.predict(X_test)

In [44]:
from sklearn.metrics import classification_report, accuracy_score

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print(f"Accuracy Score: {accuracy_score(y_test, y_pred)}")



Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.98      0.96      1200
           1       0.97      0.92      0.94       800

    accuracy                           0.96      2000
   macro avg       0.96      0.95      0.95      2000
weighted avg       0.96      0.96      0.96      2000

Accuracy Score: 0.956
